# Telco Dataset Integration

## 1. Orchestration boundary declaration and responsibility

This is the canonical, single human-facing Atlas dataset-integration entrypoint
for `telco-customer-churn` (Project Spec S0179/S0184, migrated to the
Atlas-native fixed-configuration lifecycle by Project Spec S0260). It
orchestrates Atlas input verification, dataset-specific semantic authoring,
capability-aware execution-contract materialization, Atlas-native
fixed-configuration training, native analytical evidence materialization,
inference-bundle materialization, release-candidate assembly, publisher
structural validation, conditional manifest generation, and one explicit
validated-run terminal outcome.

It remains an orchestrator: reusable generic implementation logic lives in
`pipeline/` and `publisher/` modules, never in notebook cells. It stops
unconditionally before publisher promotion, registry activation, or runtime
prediction. This notebook no longer requires, reads, or reruns any external
scientific project checkout at execution time; the historical study that
originally informed the frozen `hist_gradient_boosting` configuration is
documented as provenance context only (Section 7), never as an executable
dependency. The historical `01_dataset_authoring.ipynb` remains read-only
provenance.


In [1]:
ORCHESTRATION_BOUNDARY = {
    "allowed": [
        "atlas_input_verification",
        "dataset_specific_semantic_authoring",
        "capability_profile_declaration_or_reference",
        "reviewed_native_training_policy_authoring",
        "execution_contract_materialization",
        "capability_aware_projection",
        "native_binary_fixed_configuration_training_run_materialization",
        "native_metrics_visualization_evidence_validation",
        "inference_bundle_materialization",
        "release_candidate_assembly",
        "publisher_structural_validation",
        "manifest_generation_when_structurally_permitted",
        "validated_run_terminal_outcome",
    ],
    "still_forbidden": [
        "external_scientific_project_read_at_runtime",
        "external_model_load",
        "model_fitting_or_retraining_outside_governed_entrypoint",
        "model_selection",
        "threshold_optimization",
        "model_deserialization_or_inference_execution",
        "publisher_promotion",
        "registry_active_release_mutation",
        "public_visibility_or_profile_activation",
    ],
    "external_scientific_project_used_as_implementation_reference_only": True,
    "durable_absolute_external_path": False,
    "stops_before_promotion_registry_activation_and_runtime_prediction": True,
}
assert ORCHESTRATION_BOUNDARY["durable_absolute_external_path"] is False
assert ORCHESTRATION_BOUNDARY["stops_before_promotion_registry_activation_and_runtime_prediction"] is True
assert set(ORCHESTRATION_BOUNDARY["allowed"]).isdisjoint(ORCHESTRATION_BOUNDARY["still_forbidden"])


## 2. Dataset/source input identity

All durable Atlas inputs and outputs are repository-relative. The Atlas
repository root is resolved through the existing generic resolver
(`pipeline.discovery_evidence.resolve_repository_root`), never assigned as
`Path.cwd().resolve()`, so this notebook resolves Atlas correctly whether the
Jupyter working directory is the repository root, the notebook directory, or
another descendant the resolver can walk up from. No external scientific
project root is read, resolved, or persisted by this notebook.


In [2]:
from pathlib import Path
import hashlib
import json

from pipeline.discovery_evidence import resolve_repository_root, resolve_repository_path

repo_root = resolve_repository_root()
dataset_slug = "telco-customer-churn"
dataset_relative_path = "data/raw/telco-customer-churn.csv"

capability_profile_relative_path = "pipeline/capabilities/binary-predictive-classification.v1.json"
execution_contract_relative_path = "contracts/telco-customer-churn/execution-contract.json"
execution_contract_evidence_relative_path = "contracts/telco-customer-churn/execution-contract-materialization-evidence.json"
runtime_contract_relative_path = "contracts/telco-customer-churn/runtime-contract.json"
public_contract_relative_path = "contracts/telco-customer-churn/public-contract.json"
dataset_context_relative_path = "contracts/telco-customer-churn/dataset-context.json"
discovery_evidence_relative_path = "pipeline/evidence/telco-customer-churn/discovery-evidence.json"
prepared_data_metadata_relative_path = "pipeline/prepared/telco-customer-churn/prepared-data-metadata.json"
authoring_root_relative_path = "pipeline/authoring/telco-customer-churn"
authoring_generation_id = "telco-authoring-v2"
canonical_notebook_ref = "notebooks/datasets/telco-customer-churn/dataset_integration.ipynb"
generated_at = "2026-08-25T00:00:00+00:00"

run_state = {"blocked": False, "reasons": []}


def record_block(code_, message, field=None):
    reason = {"code": code_, "message": message}
    if field is not None:
        reason["field"] = field
    run_state["blocked"] = True
    run_state["reasons"].append(reason)
    return reason


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()


def write_governed_json(relative_path, payload):
    path = repo_root / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {"path": relative_path, "sha256": sha256_file(path)}


## 3. Atlas-owned source verification and drift checks

These observations describe what Atlas sees in the exact current, Atlas-local
raw CSV. They are not imported scientific conclusions. Absence of the
Atlas-local raw source fails this cell closed with an explicit `AssertionError`
before any later stage runs; no network/download fallback exists anywhere in
this notebook.


In [3]:
from pipeline.discovery_evidence import (
    load_dataset_csv, observe_authoring_fields, resolve_repository_path,
    summarize_structure, summarize_target_column, summarize_identifier_columns,
)
dataset_path = resolve_repository_path(dataset_relative_path, repo_root=repo_root)
rows = load_dataset_csv(dataset_path)
atlas_structure = summarize_structure(rows)
assert atlas_structure["row_count"] == 7043
assert atlas_structure["column_count"] == 21
atlas_field_observations = observe_authoring_fields(rows, atlas_structure["ordered_columns"])
atlas_target_observation = summarize_target_column(rows, "Churn")
atlas_identifier_observation = summarize_identifier_columns(rows, ["customerID"])
assert set(atlas_target_observation["observed_labels"]) == {"No", "Yes"}
assert atlas_identifier_observation[0]["is_unique_per_row"]


## 4. Dataset-specific semantic interpretation

Telco-specific field meaning, inclusion decisions, missing-value intent,
target semantics, and public meaning are materialized in
`dataset-semantic-intent.v1`. The authoring rationale references only
Atlas-local source-verification evidence; no external scientific evidence
reference is carried.


In [4]:
feature_names = [name for name in atlas_structure["ordered_columns"] if name not in {"customerID", "Churn"}]
field_role_decisions = [{"field_name": "customerID", "role": "identifier", "include_in_features": False}]
field_role_decisions += [{
    "field_name": name, "role": "feature", "include_in_features": True,
    "missing_value_intent": ({"policy": "impute_fixed_value", "fixed_value": 0.0, "rationale": "Blank TotalCharges occurs with zero tenure."} if name == "TotalCharges" else {"policy": "no_missing_expected"}),
} for name in feature_names]
field_role_decisions.append({"field_name": "Churn", "role": "target", "include_in_features": False, "exclusion_reason": "Binary result field."})
semantic_intent = {
    "schema_version": "dataset-semantic-intent.v1", "artifact_type": "dataset_semantic_intent",
    "dataset_identity": {"dataset_slug": dataset_slug, "dataset_logical_name": "Telco Customer Churn"},
    "authoring_generation_id": authoring_generation_id,
    "governing_capability_profile": {"capability_profile_id": "binary-predictive-classification", "capability_profile_version": "v1"},
    "field_role_decisions": field_role_decisions,
    "target_semantics": {"target_field_name": "Churn", "task_type": "binary_classification", "positive_class": {"class_id": "Yes", "event_label": "customer churned"}, "is_final_training_configuration": False},
    "authored_public_meaning": {"human_reviewed": True, "safe_projection_intent": "Estimate customer churn propensity from reviewed service and account fields."},
    "authoring_rationale_refs": [{"reference_kind": "source_verification_evidence", "reference_id": "atlas-current-source-observation"}],
    "semantic_boundary_confirmations": {"observed_source_statistics_embedded": False, "scientific_conclusions_embedded": False, "training_outcome_embedded": False, "release_state_embedded": False, "model_bytes_embedded": False},
    "generated_at": generated_at,
}


## 5. Capability-profile resolution

The concrete, generic Atlas-owned capability profile is read and hashed from
`pipeline/capabilities/binary-predictive-classification.v1.json`. It is never
duplicated as dataset-specific content.


In [5]:
capability_profile_path = repo_root / capability_profile_relative_path
capability_profile = json.loads(capability_profile_path.read_text(encoding="utf-8"))
assert capability_profile["schema_version"] == "capability-profile.v1"
assert capability_profile["capability_profile_id"] == "binary-predictive-classification"
assert capability_profile["capability_profile_version"] == "v1"
assert capability_profile["support_status"] == "current_supported"


## 6. Deterministic preparation/input policy

The governed preparation role records the reviewed, deterministic
`TotalCharges` rule and ordered source fields; it does not perform downstream
modeling work. The governed prepared candidate and its
`prepared-data-metadata.v1` artifact were already materialized by the
Atlas-owned `pipeline/prepare_candidate.py` producer against this exact rule
(Project Spec S0028) -- this notebook reads and re-verifies that governed
prepared candidate rather than re-deriving it inline.


In [6]:
preparation_policy = {
    "schema_version": "candidate-preparation-recipe.v1",
    "dataset_slug": dataset_slug,
    "source_data_ref": dataset_relative_path,
    "ordered_input_columns": atlas_structure["ordered_columns"],
    "transformations": [{"field": "TotalCharges", "operation": "conditional_blank_to_zero", "when": {"field": "tenure", "equals": "0"}, "otherwise": "reject_blank"}],
    "deterministic": True,
}


## 7. Reviewed native training-policy and binary result-semantics authoring

A single approved, frozen `hist_gradient_boosting` training-policy intent
(Project Spec S0257/S0258/S0259) and a single approved binary
result-semantics intent are authored here. Both are reviewed authoring intent
only -- neither is executable until materialized into the execution contract
in Section 10. The frozen hyperparameters, decision threshold, positive
class, and risk-interpretation bands reproduce this dataset's existing
governed values exactly; no model selection, hyperparameter search, or
threshold optimization is performed anywhere in this notebook. The historical
`dataset-study-telco-customer-churn` scientific study originally informed
this frozen configuration -- that is provenance context only, not an
executable dependency of this cell.


In [7]:
from pipeline.discovery_evidence import build_binary_result_semantics_intent

training_policy_intent = {
    "review_status": "approved",
    "numeric_handling": "standardize",
    "categorical_encoding_policy": "onehot",
    "allowed_transformations": ["passthrough"],
    "split_policy": {
        "strategy": "stratified",
        "train_ratio": 0.70,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
    },
    "primary_metric": "roc_auc",
    "secondary_metrics": ["f1", "pr_auc"],
    "modeling_constraints": {
        "allowed_model_families": ["hist_gradient_boosting"],
        "no_automl": True,
        "selection_mode": "fixed_configuration",
        "fixed_model_configuration": {
            "model_family": "hist_gradient_boosting",
            "hyperparameters": {
                "class_weight": None,
                "l2_regularization": 1.0,
                "learning_rate": 0.03,
                "max_iter": 200,
                "max_leaf_nodes": 7,
                "min_samples_leaf": 40,
                "max_depth": 3,
            },
        },
    },
}
assert training_policy_intent["review_status"] == "approved"

binary_result_semantics_intent = build_binary_result_semantics_intent(
    review_status="approved",
    problem_type="binary_classification",
    positive_class_id="Yes",
    event_label="Churn",
    primary_output="positive_class_probability",
    threshold=0.5,
    preset="risk",
    bands=[
        {"band_id": "low", "lower_bound": 0.0, "upper_bound": 0.35},
        {"band_id": "medium", "lower_bound": 0.35, "upper_bound": 0.65},
        {"band_id": "high", "lower_bound": 0.65, "upper_bound": 1.0},
    ],
)
assert binary_result_semantics_intent["review_status"] == "approved"


## 8. Atlas authoring artifact materialization

The semantic intent and preparation policy are durable governed artifacts.
The principal manifest coordinates them and Atlas source verification by
safe relative references and hashes. No external provenance is carried --
`provenance` is an empty list, since this authoring generation consumes no
external scientific evidence.


In [8]:
semantic_ref = write_governed_json(f"{authoring_root_relative_path}/dataset-semantic-intent.json", semantic_intent)
preparation_ref = write_governed_json(f"{authoring_root_relative_path}/preparation-recipe.json", preparation_policy)
source_verification_ref = {"path": discovery_evidence_relative_path, "sha256": sha256_file(repo_root / discovery_evidence_relative_path)}
artifact_references = [
    {"role": "discovery_evidence", **source_verification_ref, "contract_version": "dataset-discovery-evidence.v1"},
    {"role": "semantic_intent", **semantic_ref, "contract_version": "dataset-semantic-intent.v1"},
    {"role": "preparation_recipe", **preparation_ref, "contract_version": "candidate-preparation-recipe.v1"},
]
manifest = {
    "schema_version": "dataset-integration-authoring-manifest.v1", "artifact_type": "dataset_integration_authoring_manifest",
    "dataset_identity": {"dataset_slug": dataset_slug, "dataset_logical_name": "Telco Customer Churn"},
    "authoring_generation": {"authoring_generation_id": authoring_generation_id, "immutable": True, "generated_at": generated_at},
    "capability_profile_selection": {"capability_profile_id": capability_profile["capability_profile_id"], "capability_profile_version": capability_profile["capability_profile_version"], "capability_profile_ref": {"path": capability_profile_relative_path, "sha256": sha256_file(capability_profile_path)}},
    "artifact_references": artifact_references,
    "provenance": [],
    "boundary_confirmations": {"complete_discovery_evidence_embedded": False, "complete_semantic_intent_embedded": False, "complete_preparation_recipe_embedded": False, "training_metrics_embedded": False, "model_selection_payload_embedded": False, "model_bytes_embedded": False, "inference_bundle_payload_embedded": False, "visual_payloads_embedded": False, "absolute_external_project_root_present": False, "external_analysis_handoff_replacement": False, "operational_importer_instruction_present": False},
    "generated_at": generated_at,
}
manifest_ref = write_governed_json(f"{authoring_root_relative_path}/dataset-integration-authoring-manifest.json", manifest)


## 9. Cross-artifact authoring validation

The generic S0166 validator checks schemas, identities, capability
applicability, safe paths, and referenced hashes.


In [9]:
from pipeline.authoring_contracts import validate_authoring_contracts
authoring_validation = validate_authoring_contracts(manifest, capability_profile, semantic_intent=semantic_intent, artifact_root=repo_root, expected_dataset_slug=dataset_slug, generated_at=generated_at)
assert authoring_validation.valid, authoring_validation.failures


## 10. Capability-aware execution contract materialization

The reviewed training-policy and binary result-semantics intents from
Section 7 are materialized into a fresh, schema-valid `execution_contract.v1`
through the existing canonical, dataset-agnostic derivation module
(`pipeline.contract_derivation.materialize_execution_contract`) -- never a
hand-authored contract document. The materialized contract declares
`selection_mode=fixed_configuration` and
`fixed_model_configuration.model_family=hist_gradient_boosting`, and never
declares `model_source_mode=validated_external_fitted_model`.


In [10]:
from pipeline.discovery_evidence import build_dataset_modeling_intent
from pipeline.contract_derivation import materialize_execution_contract

atlas_discovery_evidence = json.loads((repo_root / discovery_evidence_relative_path).read_text(encoding="utf-8"))

modeling_intent = build_dataset_modeling_intent(
    dataset_slug=dataset_slug,
    dataset_source_ref=dataset_relative_path,
    authoring_notebook_ref=canonical_notebook_ref,
    columns=atlas_structure["ordered_columns"],
    target_column="Churn",
    task_type="classification",
    observed_labels=atlas_target_observation["observed_labels"],
    positive_label_candidate="Yes",
    observed_target_distribution=atlas_target_observation["observed_distribution"],
    identifier_columns=["customerID"],
    training_policy_intent=training_policy_intent,
    binary_result_semantics_intent=binary_result_semantics_intent,
    reduced_discovery_evidence_ref=discovery_evidence_relative_path,
    generated_at=generated_at,
)

execution_contract_materialization = materialize_execution_contract(
    modeling_intent,
    atlas_discovery_evidence,
    execution_contract_relative_path,
    repo_root,
    preparation_recipe=preparation_policy,
    evidence_output_relative_path=execution_contract_evidence_relative_path,
    discovery_evidence_relative_path=discovery_evidence_relative_path,
    preparation_recipe_relative_path=preparation_ref["path"],
    prepared_data_metadata_relative_path=prepared_data_metadata_relative_path,
    raw_dataset_relative_path=dataset_relative_path,
    semantic_intent=semantic_intent,
    generated_at=generated_at,
)
execution_contract = execution_contract_materialization["execution_contract"]
assert execution_contract["modeling_constraints"]["selection_mode"] == "fixed_configuration"
assert execution_contract["modeling_constraints"]["fixed_model_configuration"]["model_family"] == "hist_gradient_boosting"
assert "model_source_mode" not in execution_contract
assert execution_contract["result_semantics"]["schema_version"] == "binary-result-semantics.v1"
assert execution_contract["result_semantics"]["positive_class"]["class_id"] == "Yes"
assert execution_contract["result_semantics"]["decision"]["threshold"] == 0.5


## 11. Runtime/public contract and dataset-context compatibility confirmation

The existing, already-governed `runtime-contract.json`, `public-contract.json`,
and `dataset-context.json` (carrying the existing Telco Predict View) remain
compatible with the freshly materialized execution contract's feature
identity -- confirmed here explicitly, never silently assumed. This notebook
never rewrites these three files.


In [11]:
runtime_contract = json.loads((repo_root / runtime_contract_relative_path).read_text(encoding="utf-8"))
public_contract = json.loads((repo_root / public_contract_relative_path).read_text(encoding="utf-8"))
dataset_context = json.loads((repo_root / dataset_context_relative_path).read_text(encoding="utf-8"))

assert [f["name"] for f in runtime_contract["features"]] == execution_contract["feature_columns"]
assert [f["name"] for f in public_contract["features"]] == execution_contract["feature_columns"]
assert dataset_context["dataset_slug"] == dataset_slug
assert len(dataset_context.get("predict_views", [])) == 1


## 12. Native training readiness

The generic training-readiness gate independently re-confirms the
just-materialized execution contract is execution-ready before any training
call is attempted.


In [12]:
from pipeline.training import prepare_training_invocation_readiness

training_readiness = prepare_training_invocation_readiness(
    repo_root / execution_contract_relative_path,
    repo_root / dataset_relative_path,
)
if not training_readiness["is_training_ready"]:
    for reason in training_readiness["blocking_reasons"]:
        record_block("training_not_ready", reason)
assert training_readiness["execution_contract_identity"] == "execution_ready"


## 13. Native binary fixed-configuration training run materialization

The governed generic entrypoint `pipeline.training.train_from_paths` (via
`pipeline.training.materialize_training_run_from_prepared_metadata`, which
resolves the already-governed, already-training-ready prepared candidate and
then calls `train_from_paths` itself) performs the one frozen
`hist_gradient_boosting` fit. No model selection, hyperparameter search, or
threshold optimization occurs -- the fixed-configuration training path never
touches the historical multi-candidate selection code.


In [13]:
from pipeline import training as pipeline_training

if not run_state["blocked"]:
    training_run_materialization_result = pipeline_training.materialize_training_run_from_prepared_metadata(
        repo_root / execution_contract_relative_path,
        repo_root / prepared_data_metadata_relative_path,
        dataset_slug=dataset_slug,
    )
    if training_run_materialization_result["status"] != "trained":
        for reason in training_run_materialization_result["blocking_reasons"]:
            record_block("native_training_blocked", reason)


## 14. Native metrics/visualization evidence validation

The real fitted model's own governed classification evidence -- never a
notebook-side literal -- is the authority for positive-class identity and
class-label order. `training-metrics.v5` and `analytical-visualizations.v5`
are required; a sealed single test evaluation is confirmed complete.


In [14]:
if not run_state["blocked"]:
    training_result = training_run_materialization_result["training_result"]
    training_run_relative_path = training_result["output_directory"]
    training_parameter_record = json.loads(
        (repo_root / training_result["training_parameter_record_path"]).read_text(encoding="utf-8")
    )
    assert training_parameter_record["schema_version"] == "training-parameter-record.v5"
    real_fitted_classification_evidence = training_parameter_record["classification_evidence"]
    real_fitted_class_order = real_fitted_classification_evidence["ordered_class_labels"]
    assert set(real_fitted_class_order) == {"No", "Yes"}
    # The real fitted model's own positive-class resolution is the technical
    # authority -- verified here to agree with the governed execution
    # contract rather than ever being silently overridden.
    assert real_fitted_classification_evidence["positive_class_id"] == "Yes"

    training_metrics = json.loads((repo_root / training_result["metrics_path"]).read_text(encoding="utf-8"))
    assert training_metrics["schema_version"] == "training-metrics.v5"
    assert training_metrics["final_test_evaluation"]["completed"] is True
    assert training_metrics["final_test_evaluation"]["evaluation_count"] == 1

    analytical_visualizations = json.loads(
        (repo_root / training_result["analytical_visualizations_path"]).read_text(encoding="utf-8")
    )
    assert analytical_visualizations["schema_version"] == "analytical-visualizations.v5"
    assert analytical_visualizations["classification_evidence"]["positive_class_id"] == "Yes"


## 15. Governed inference-bundle generation

The generic bundle producer is called through its internal-training branch
only, using the real training-run materialization result -- never an
external fitted-model materialization result.


In [15]:
from pipeline import generate_inference_bundle, release_identity

inference_bundle_relative_path = ""

allocated_release_id = None

if not run_state["blocked"]:
    run_id = Path(training_run_relative_path.rstrip("/")).name
    allocated_release_id = release_identity.allocate_release_id(run_id, repo_root)
    inference_bundle_relative_path = f"pipeline/inference-bundles/{dataset_slug}/inference-bundle.json"

    inference_bundle_result = generate_inference_bundle.materialize_governed_inference_bundle(
        training_run_materialization_result=training_run_materialization_result,
        execution_contract_path=repo_root / execution_contract_relative_path,
        runtime_contract_path=repo_root / runtime_contract_relative_path,
        public_contract_path=repo_root / public_contract_relative_path,
        dataset_context_path=repo_root / dataset_context_relative_path,
        prepared_data_metadata_path=repo_root / prepared_data_metadata_relative_path,
        output_path=repo_root / inference_bundle_relative_path,
        prediction_type="number",
        repo_root=repo_root,
        dataset_slug=dataset_slug,
        class_labels=real_fitted_class_order,
        probability_output=True,
        execution_contract_ref=execution_contract_relative_path,
        runtime_contract_ref=runtime_contract_relative_path,
        public_contract_ref=public_contract_relative_path,
        dataset_context_ref=dataset_context_relative_path,
        model_package_reference="models/model.pkl",
        release_id=allocated_release_id,
    )
    if inference_bundle_result["status"] != "generated":
        for reason in inference_bundle_result["blocking_reasons"]:
            record_block("inference_bundle_blocked", reason)

if not run_state["blocked"]:
    inference_bundle = json.loads((repo_root / inference_bundle_relative_path).read_text(encoding="utf-8"))
    assert inference_bundle["release_context"]["release_id"] == allocated_release_id
    assert inference_bundle["result_semantics"]["schema_version"] == "binary-result-semantics.v1"
    assert inference_bundle["result_semantics"]["positive_class"]["class_id"] == "Yes"
    assert inference_bundle["result_semantics"]["decision"]["threshold"] == 0.5
    assert inference_bundle["result_semantics"]["model_descriptor"]["model_family"] == "hist_gradient_boosting"


## 16. Release-candidate assembly from compatible governed roles

Uses the existing generic candidate primitives
(`pipeline.assemble_candidate.build_release_candidate_input`,
`pipeline.assemble_candidate.assemble_release_candidate`). Every role
references a real, current-run, Atlas-native artifact -- the model card is
the one the native training run itself produced, never a borrowed or
fabricated substitute.


In [16]:
from pipeline import assemble_candidate

release_candidate_assembly_result = None

if not run_state["blocked"]:
    candidate_artifact_references = {
        "discovery_evidence": discovery_evidence_relative_path,
        "execution_contract": execution_contract_relative_path,
        "runtime_contract": runtime_contract_relative_path,
        "public_contract": public_contract_relative_path,
        "preparation_recipe": preparation_ref["path"],
        "prepared_data_metadata": prepared_data_metadata_relative_path,
        "training_parameter_record": training_result["training_parameter_record_path"],
        "model_artifact": training_result["serialized_model_path"],
        "training_metrics": training_result["metrics_path"],
        "model_card": training_result["model_card_path"],
        "public_context": dataset_context_relative_path,
        "visualizations": training_result["analytical_visualizations_path"],
        "inference_bundle": inference_bundle_relative_path,
    }

    candidate_handoff_readiness = assemble_candidate.build_release_candidate_handoff_readiness(
        candidate_artifact_references, repo_root=repo_root
    )
    if candidate_handoff_readiness["is_release_candidate_input_ready"]:
        candidate_input = assemble_candidate.build_release_candidate_input(
            dataset_slug=dataset_slug,
            release_id=allocated_release_id,
            source_run_id=run_id,
            artifact_references=candidate_artifact_references,
            repo_root=repo_root,
            release_version="1.0.0-rc.1",
            dataset_title="Telco Customer Churn",
        )
        release_candidate_assembly_result = assemble_candidate.assemble_release_candidate(
            candidate_input,
            repo_root / "releases" / "candidates",
            repo_root=repo_root,
        )
        if release_candidate_assembly_result.get("status") != "accepted":
            record_block(
                "candidate_assembly_rejected",
                f"release-candidate assembly did not reach accepted status: {release_candidate_assembly_result.get('reason')}",
            )
    else:
        for unready_role in candidate_handoff_readiness["not_ready_roles"]:
            unready_role_result = next(
                (r for r in candidate_handoff_readiness["role_results"] if r["role"] == unready_role),
                None,
            )
            unready_reason = unready_role_result.get("reason") if unready_role_result else None
            record_block(
                "candidate_role_unavailable",
                f"release-candidate handoff readiness failed for required role {unready_role!r}"
                + (f": {unready_reason}" if unready_reason else "."),
                unready_role,
            )


## 17. Publisher Run materialization

Uses `publisher.validate.materialize_validation_run`, the modern,
dataset-agnostic Publisher Run materializer, rather than filesystem scanning
around `publisher.validate.run` and manual manifest orchestration. For an
accepted candidate this materializes exactly one Publisher Run directory
carrying `validation-result.json` and (when structurally permitted)
`manifest.json`.


In [17]:
from publisher import validate as publisher_validate

publisher_materialization_result = None
publisher_run_id = None
publisher_run_dir_relative_path = None
publisher_validation_outcome = None
publisher_manifest_relative_path = None

if not run_state["blocked"] and release_candidate_assembly_result is not None:
    publisher_materialization_result = publisher_validate.materialize_validation_run(
        release_candidate_assembly_result, repo_root=repo_root,
    )

    if publisher_materialization_result["materialization_status"] != "materialized":
        record_block(
            "publisher_run_materialization_blocked",
            publisher_materialization_result.get("message")
            or f"publisher run materialization blocked: {publisher_materialization_result.get('reason_code')}",
        )
    else:
        publisher_run_id = publisher_materialization_result["run_id"]
        publisher_run_dir_relative_path = publisher_materialization_result["run_dir"]
        publisher_validation_outcome = publisher_materialization_result["validation_outcome"]
        publisher_manifest_relative_path = publisher_materialization_result["manifest_path"]
        publisher_run_dir = repo_root / publisher_run_dir_relative_path

        if publisher_validation_outcome != "accepted":
            record_block(
                "publisher_structural_validation_rejected",
                "publisher.validate did not accept the assembled candidate.",
            )
        elif not publisher_materialization_result["manifest_generated"]:
            record_block(
                "publisher_manifest_not_generated",
                publisher_materialization_result.get("manifest_error")
                or "manifest was not generated for an accepted Publisher Run.",
            )
        elif not publisher_run_dir.is_dir():
            record_block("publisher_run_directory_missing", "Publisher Run directory does not exist.")
        elif not (publisher_run_dir / "validation-result.json").is_file():
            record_block(
                "publisher_validation_result_missing",
                "validation-result.json is missing from the Publisher Run directory.",
            )
        elif not (publisher_run_dir / "manifest.json").is_file():
            record_block(
                "publisher_manifest_file_missing",
                "manifest.json is missing from the Publisher Run directory.",
            )


## 18. Validated-run terminal result

Exactly one explicit validated-run terminal outcome is materialized through
`pipeline.validated_run.materialize_validated_run_terminal_result`, declaring
`model_source_mode=atlas_internal_training`. This notebook never reimplements
promotion-eligibility logic; the generic terminal producer owns the
eligibility/hash/schema decisions. Atlas-native internal training never
carries an external operational-readiness profile.


In [18]:
from pipeline import validated_run


def _durable_ref(path_value):
    if path_value is None:
        return None
    return {"path": path_value, "sha256": sha256_file(repo_root / path_value)}


validated_run_terminal_result = None
terminal_result_ref = None

if publisher_run_dir_relative_path is not None:
    release_candidate_json_relative_path = None
    if release_candidate_assembly_result is not None and release_candidate_assembly_result.get("status") == "accepted":
        release_candidate_json_relative_path = (
            f"{release_candidate_assembly_result['candidate_dir']}/release-candidate.json".replace(str(repo_root) + "/", "")
        )

    durable_references = {
        "materialization_result": None,
        "inference_bundle": _durable_ref(
            inference_bundle_relative_path
            if inference_bundle_relative_path and inference_bundle_result.get("status") == "generated"
            else None
        ),
        "release_candidate": _durable_ref(release_candidate_json_relative_path),
        "publisher_validation_result": _durable_ref(f"{publisher_run_dir_relative_path}/validation-result.json"),
        "manifest": _durable_ref(publisher_manifest_relative_path),
        "operational_readiness_source": None,
    }

    structural_validation = (
        {"validation_outcome": publisher_validation_outcome}
        if publisher_validation_outcome is not None
        else None
    )

    manifest_outcome = (
        {
            "manifest_generated": publisher_materialization_result["manifest_generated"],
            "manifest_path": publisher_manifest_relative_path,
        }
        if publisher_materialization_result is not None
        else None
    )

    operational_readiness = {
        "operational_validity": "not_applicable",
        "operational_threshold": {"status": "not_applicable", "value": None},
        "operational_prediction_available": False,
    }

    terminal_status = "blocked" if run_state["blocked"] else "completed"
    terminal_reasons = run_state["reasons"] if run_state["blocked"] else None

    validated_run_terminal_result = validated_run.materialize_validated_run_terminal_result(
        run_id=run_id,
        dataset_slug=dataset_slug,
        model_source_mode="atlas_internal_training",
        status=terminal_status,
        durable_references=durable_references,
        structural_validation=structural_validation,
        manifest_outcome=manifest_outcome,
        operational_readiness=operational_readiness,
        reasons=terminal_reasons,
        repo_root=repo_root,
    )

    terminal_result_relative_path = f"{publisher_run_dir_relative_path}/validated-run-terminal-result.json"
    terminal_result_ref = write_governed_json(terminal_result_relative_path, validated_run_terminal_result)

    assert validated_run_terminal_result["status"] in ("completed", "blocked", "failed")
    assert validated_run_terminal_result["promotion_eligibility"] in (True, False)
    if validated_run_terminal_result["status"] == "completed":
        assert validated_run_terminal_result["promotion_eligibility"] is True


## 19. Orchestration stop confirmation

This notebook's terminal artifact is `validated_run_terminal_result`. It
contains no active call that can run `publisher.promote.run`, mutate
`registry/datasets.json` `active_release`, activate a release, change public
visibility, write profile publication state, load or deserialize the fitted
model outside the governed training entrypoint, or serve/predict through
runtime inference. `promotion_eligibility: true`, if produced, is
informational only inside this notebook -- promotion remains a separate,
operator-controlled work package.


In [19]:
orchestration_summary = {
    "dataset_slug": dataset_slug,
    "canonical_notebook_ref": canonical_notebook_ref,
    "run_blocked": run_state["blocked"],
    "blocking_reasons": run_state["reasons"],
    "publisher_run_id": publisher_run_id,
    "publisher_run_dir": publisher_run_dir_relative_path,
    "terminal_status": validated_run_terminal_result["status"] if validated_run_terminal_result else None,
    "promotion_eligibility": (
        validated_run_terminal_result["promotion_eligibility"] if validated_run_terminal_result else False
    ),
    "stops_before_promotion_registry_activation_and_runtime_prediction": True,
    "promotion_performed": False,
    "registry_activation_performed": False,
    "public_visibility_or_profile_activation_performed": False,
}
orchestration_summary


{'dataset_slug': 'telco-customer-churn',
 'canonical_notebook_ref': 'notebooks/datasets/telco-customer-churn/dataset_integration.ipynb',
 'run_blocked': False,
 'blocking_reasons': [],
 'publisher_run_id': 'validate-20260829T144328Z',
 'publisher_run_dir': 'publisher/runs/validate-20260829T144328Z',
 'terminal_status': 'completed',
 'promotion_eligibility': True,
 'stops_before_promotion_registry_activation_and_runtime_prediction': True,
 'promotion_performed': False,
 'registry_activation_performed': False,
 'public_visibility_or_profile_activation_performed': False}